**Github** : https://github.com/OzgurYldrm/AI-ML-Course     
**Youtube** : https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence

https://tr.wikipedia.org/wiki/Vikipedi:Veritaban%C4%B1_indirme

In [1]:
import re
import pandas as pd
from collections import defaultdict
import math

# Data

In [2]:
df = pd.read_csv("trwiki_random_2000.csv")

In [3]:
def wiki_clean(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"#YÖNLENDİRME.*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<.*?>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<ref.*?>.*?</ref>", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\{.*?\}\}", " ", text, flags=re.DOTALL)
    text = re.sub(r"\{\|.*?\|\}", " ", text, flags=re.DOTALL)
    text = re.sub(r"\[\[File:.*?\]\]", " ", text)
    text = re.sub(r"\[\[Dosya:.*?\]\]", " ", text)
    text = re.sub(r"\[\[.*?\|(.*?)\]\]", r"\1", text)
    text = re.sub(r"\[\[.*?\]\]", " ", text)
    text = text.replace("'''", " ")
    text = text.replace("''", " ")
    text = re.sub(r"\|.*?=", " ", text)
    text = re.sub(r"==.*?==", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ\s]", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


In [4]:
def tokenize(text):
    return text.split()

def prepare_sentence(sentence):
    tokens = tokenize(sentence)
    return ["<s>", "<s>"] + tokens + ["</s>"]

In [5]:
df["text"] = df["text"].apply(wiki_clean)
df["text"] = df["text"].apply(prepare_sentence)

In [6]:
df

,title,text
0,Sedum patrickii,"[<s>, <s>, sedum, patrickii, cinsine, bağlı, b..."
1,Caselle in Pittari,"[<s>, <s>, ülke, caselle, in, pittari, şehrini..."
2,Dekan Yaylası,"[<s>, <s>, </s>]"
3,Pat O'Brien,"[<s>, <s>, müzisyen, şarkı, sözü, yazarı, gita..."
4,Musée de l'Aventure Peugeot,"[<s>, <s>, mus, e, de, l, aventure, peugeot, p..."
...,...,...
1995,Vikipedi:Seçkin resim adayları/Plagiomnium aff...,"[<s>, <s>, mach, iavelli, msj, mart, utc, thum..."
1996,Karanlıkta Bir Çığlık (anlam ayrımı),"[<s>, <s>, blake, edwards, ın, yönettiği, yapı..."
1997,Kategori:Gambiya'daki çevre,"[<s>, <s>, çevre, </s>]"
1998,Kategori:Original Memphis Five üyeleri,"[<s>, <s>, </s>]"


# Model

In [7]:
trigram_counts = defaultdict(lambda: defaultdict(int))
bigram_counts = defaultdict(int)

for tokens in df["text"]:
    for i in range(2, len(tokens)):
        w1, w2, w3 = tokens[i-2], tokens[i-1], tokens[i]
        
        trigram_counts[(w1, w2)][w3] += 1
        bigram_counts[(w1, w2)] += 1

In [10]:
vocab = set()
for tokens in df["text"]:
    for token in tokens:
        vocab.add(token)
V = len(vocab)

In [12]:
print("Vocabulary size:", V)
print("Toplam trigram çeşidi:", len(trigram_counts))

Vocabulary size: 20807
Toplam trigram çeşidi: 53705


In [13]:
def trigram_prob(w1, w2, w3):
    context_count = bigram_counts[(w1, w2)]
    word_count = trigram_counts[(w1, w2)][w3]
    return (word_count + 1) / (context_count + V)

In [16]:
print(trigram_prob("bir","bitki","türüdür"))
print(trigram_prob("bir","bitki","yapay"))

0.0019184652278177458
4.796163069544364e-05


In [17]:
def sentence_log_prob(tokens):
    log_prob = 0.0
    for i in range(2, len(tokens)):
        w1, w2, w3 = tokens[i-2], tokens[i-1], tokens[i]
        p = trigram_prob(w1, w2, w3)
        log_prob += math.log(p)
    return log_prob

In [18]:
sentence_log_prob("Bir bitki türüdür".split())

-9.943044747534582

In [19]:
def top_k_next(w1, w2, k=5):
    context_count = bigram_counts[(w1, w2)]
    if context_count == 0:
        return [(word, 1 / V) for word in list(vocab)[:k]]
    candidates = []
    for word, count in trigram_counts[(w1, w2)].items():
        prob = (count + 1) / (context_count + V)
        candidates.append((word, prob))
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k]

In [24]:
top_k_next("bir","bitki",k=5)

[('türüdür', 0.0019184652278177458),
 ('üdür', 0.00014388489208633093),
 ('idir', 9.592326139088728e-05),
 ('türü', 9.592326139088728e-05),
 ('yapay', 4.796163069544364e-05)]

In [25]:
def greedy_next(w1, w2):
    context_count = bigram_counts[(w1, w2)]
    if context_count == 0:
        return "</s>"
    best_word = None
    best_prob = 0
    for word, count in trigram_counts[(w1, w2)].items():
        prob = (count + 1) / (context_count + V)
        if prob > best_prob:
            best_prob = prob
            best_word = word
    return best_word

def generate_text(max_tokens=20,generated=["<s>", "<s>"]):
    while len(generated) < max_tokens + 2:
        w1, w2 = generated[-2], generated[-1]
        next_word = greedy_next(w1, w2)
        if next_word == "</s>" or next_word is None:
            break
        generated.append(next_word)
    return " ".join(generated[2:]) # padding tokenları çıkar

In [26]:
generate_text(20)

''

In [40]:
generate_text(20, "<s> yıldız".split())

'süper madde yıldızı açıklama i̇bp de bayt büyüklüğünde olmalı ve en üretken ekosistemlerini oluşturmaktadır bu da aynı akıbete uğramasın iyi'